<a href="https://colab.research.google.com/github/Ashu-42/quant_research_crypto_volatility_forecasting/blob/quant_DL_ashu/notebooks/02_data_processing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Imports n Data read

In [ ]:
from sklearn.preprocessing import StandardScaler
from google.colab import drive
from pathlib import Path
import pandas as pd
import numpy as np
import json
import joblib

In [ ]:
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# read the stored raw data

shared_project_dir = Path("/content/drive/MyDrive/Quant Research")
data_dir = shared_project_dir / "data"
processed_dir = data_dir / "processed"

filename = "crypto_binance_15m_btc_eth_sol_xrp_v1.parquet"

shared_file_path = (
    processed_dir /
    filename
)

drive_df = pd.read_parquet(shared_file_path)

print(drive_df.shape)


(1112863, 19)


### Basic Check and PreProcessing

In [ ]:
drive_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1112863 entries, 0 to 1112862
Data columns (total 19 columns):
 #   Column                  Non-Null Count    Dtype              
---  ------                  --------------    -----              
 0   open_time               1112863 non-null  datetime64[ns, UTC]
 1   close_time              1112863 non-null  datetime64[ns, UTC]
 2   asset                   1112863 non-null  object             
 3   symbol                  1112863 non-null  object             
 4   open                    1112863 non-null  float64            
 5   high                    1112863 non-null  float64            
 6   low                     1112863 non-null  float64            
 7   close                   1112863 non-null  float64            
 8   base_volume             1112863 non-null  float64            
 9   quote_volume            1112863 non-null  float64            
 10  number_of_trades        1112863 non-null  int64              
 11  taker_buy_b

In [ ]:
drive_df.isna().sum()

,0
open_time,0
close_time,0
asset,0
symbol,0
open,0
high,0
low,0
close,0
base_volume,0
quote_volume,0


In [ ]:
drive_df.head()

,open_time,close_time,asset,symbol,open,high,low,close,base_volume,quote_volume,number_of_trades,taker_buy_base_volume,taker_buy_quote_volume,close_return,intraday_return,intraday_range,base_volume_return,quote_volume_return,log_return
0,2017-08-17 04:00:00+00:00,2017-08-17 04:14:59.999000+00:00,BTC,BTCUSDT,4261.48,4280.56,4261.48,4261.48,2.189061,9333.620962,9,0.489061,2089.104962,NaN,0.000000,0.004477,NaN,NaN,NaN
1,2017-08-17 04:00:00+00:00,2017-08-17 04:14:59.999000+00:00,ETH,ETHUSDT,301.13,301.13,298.00,298.00,5.801670,1744.766104,22,5.483920,1649.448824,NaN,-0.010394,0.010394,NaN,NaN,NaN
2,2017-08-17 04:15:00+00:00,2017-08-17 04:29:59.999000+00:00,BTC,BTCUSDT,4261.48,4270.41,4261.32,4261.45,9.119865,38891.133046,40,3.447113,14703.934995,-0.000007,-0.000007,0.002133,3.166108,3.166779,-0.000007
3,2017-08-17 04:15:00+00:00,2017-08-17 04:29:59.999000+00:00,ETH,ETHUSDT,298.00,300.80,298.00,299.39,31.440650,9396.917813,26,12.117120,3625.167051,0.004664,0.004664,0.009396,4.419241,4.385775,0.004654
4,2017-08-17 04:30:00+00:00,2017-08-17 04:44:59.999000+00:00,BTC,BTCUSDT,4280.00,4310.07,4267.99,4310.07,21.923552,94080.917568,58,20.421317,87620.977876,0.011409,0.007026,0.009832,1.403934,1.419084,0.011345


In [ ]:
# dropping nulls

print(drive_df.shape)

# Keep rows required for OHLCV calculations.
# Zero-volume candles are valid and should not be removed.

required_columns = [
    "open_time",
    "close_time",
    "asset",
    "symbol",
    "open",
    "high",
    "low",
    "close",
    "base_volume",
]

print("Rows before validation:", drive_df.shape)

drive_df = drive_df.dropna(
    subset=required_columns
).copy()

valid_row_mask = (
    (drive_df["open"] > 0)
    & (drive_df["high"] > 0)
    & (drive_df["low"] > 0)
    & (drive_df["close"] > 0)
    & (drive_df["base_volume"] >= 0)
)

drive_df = drive_df.loc[valid_row_mask].copy()

drive_df = (
    drive_df
    .sort_values(["symbol", "open_time"])
    .drop_duplicates(
        subset=["symbol", "open_time"],
        keep="last"
    )
    .reset_index(drop=True)
)

# Keep date as a timezone-aware pandas datetime.
drive_df["date"] = drive_df["open_time"].dt.floor("D")

print("Rows after validation:", drive_df.shape)

print(drive_df.shape)

(1112863, 19)
Rows before validation: (1112863, 19)
Rows after validation: (1112863, 20)
(1112863, 20)


In [ ]:
# # dropping the days where we have incomplete values and remove the entries with 0 volume

# # drive_df = drive_df[drive_df.base_volume > 0]
# drive_df['date'] = drive_df.open_time.dt.date

# drop_df = drive_df.groupby(['date', 'asset']).size().reset_index()

# drop_df.columns = ['date', 'asset', 'num_values']

# drop_df = drop_df.loc[drop_df.num_values == 96, ['date', 'asset']]

# print(drive_df.shape)
# drive_df = pd.merge(left=drive_df, right=drop_df, how='inner', on=['date', 'asset'])
# print(drive_df.shape)

In [ ]:
# Recompute log returns after final sorting and cleaning.

previous_time = (
    drive_df.groupby("symbol")["open_time"]
    .shift(1)
)

previous_close = (
    drive_df.groupby("symbol")["close"]
    .shift(1)
)

time_gap = drive_df["open_time"] - previous_time

raw_log_return = np.log(
    drive_df["close"] / previous_close
)

# A valid return must cover exactly one 15-minute interval.
drive_df["log_return"] = raw_log_return.where(
    time_gap.eq(pd.Timedelta(minutes=15))
)

print(
    drive_df.groupby("symbol")["log_return"]
    .apply(lambda x: x.isna().sum())
)

symbol
BTCUSDT    34
ETHUSDT    34
SOLUSDT    11
XRPUSDT    27
Name: log_return, dtype: int64


In [ ]:
# checking duplicates

duplicate_mask = drive_df.duplicated(
    subset=["symbol", "open_time"],
    keep=False
)

print("Duplicate rows:", duplicate_mask.sum())

Duplicate rows: 0


In [ ]:
invalid_ohlc = (
    (drive_df["high"] < drive_df[["open", "close", "low"]].max(axis=1))
    | (drive_df["low"] > drive_df[["open", "close", "high"]].min(axis=1))
)

print("Invalid OHLC rows:", invalid_ohlc.sum())

Invalid OHLC rows: 0


In [ ]:
EXPECTED_CANDLES_PER_DAY = 96

daily_quality = (
    drive_df.groupby(["symbol", "date"])
    .agg(
        candle_count=("open_time", "size"),
        unique_candle_count=("open_time", "nunique"),
        valid_return_count=("log_return", "count"),
    )
    .reset_index()
)

daily_quality["is_complete_day"] = (
    (daily_quality["candle_count"] == EXPECTED_CANDLES_PER_DAY)
    & (
        daily_quality["unique_candle_count"]
        == EXPECTED_CANDLES_PER_DAY
    )
    & (
        daily_quality["valid_return_count"]
        == EXPECTED_CANDLES_PER_DAY
    )
)

completeness_summary = (
    daily_quality.groupby("symbol")
    .agg(
        total_days=("date", "count"),
        complete_days=("is_complete_day", "sum"),
        minimum_candles=("candle_count", "min"),
        maximum_candles=("candle_count", "max"),
    )
)

completeness_summary["complete_day_percentage"] = (
    completeness_summary["complete_days"]
    / completeness_summary["total_days"]
    * 100
)

display(completeness_summary)

,total_days,complete_days,minimum_candles,maximum_candles,complete_day_percentage
symbol,,,,,
BTCUSDT,3241,3205,1,96,98.889232
ETHUSDT,3241,3205,1,96,98.889232
SOLUSDT,2151,2139,1,96,99.442120
XRPUSDT,2981,2953,1,96,99.060718


In [ ]:
valid_days = daily_quality.loc[
    daily_quality["is_complete_day"],
    ["symbol", "date"]
]

print("Rows before complete-day filtering:", drive_df.shape)

drive_df = drive_df.merge(
    valid_days,
    on=["symbol", "date"],
    how="inner",
    validate="many_to_one"
)

drive_df = (
    drive_df
    .sort_values(["symbol", "open_time"])
    .reset_index(drop=True)
)

print("Rows after complete-day filtering:", drive_df.shape)

Rows before complete-day filtering: (1112863, 20)
Rows after complete-day filtering: (1104192, 20)


In [ ]:
drive_df.columns

Index(['open_time', 'close_time', 'asset', 'symbol', 'open', 'high', 'low',
       'close', 'base_volume', 'quote_volume', 'number_of_trades',
       'taker_buy_base_volume', 'taker_buy_quote_volume', 'close_return',
       'intraday_return', 'intraday_range', 'base_volume_return',
       'quote_volume_return', 'log_return', 'date'],
      dtype='object')

In [ ]:
drive_df.describe(include='all')

,open_time,close_time,asset,symbol,open,high,low,close,base_volume,quote_volume,number_of_trades,taker_buy_base_volume,taker_buy_quote_volume,close_return,intraday_return,intraday_range,base_volume_return,quote_volume_return,log_return,date
count,1104192,1104192,1104192,1104192,1.104192e+06,1.104192e+06,1.104192e+06,1.104192e+06,1.104192e+06,1.104192e+06,1.104192e+06,1.104192e+06,1.104192e+06,1.104192e+06,1.104192e+06,1.104192e+06,1.104158e+06,1.104158e+06,1.104192e+06,1104192
unique,NaN,NaN,4,4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
top,NaN,NaN,BTC,BTCUSDT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
freq,NaN,NaN,307680,307680,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
mean,2022-06-14 18:40:08.215964160+00:00,2022-06-14 18:55:08.214956544+00:00,NaN,NaN,1.115316e+04,1.117525e+04,1.113034e+04,1.115320e+04,8.968250e+05,9.176601e+06,1.290588e+04,4.445200e+05,4.542774e+06,1.913416e-05,1.989283e-05,6.144007e-03,inf,inf,5.515464e-06,2022-06-14 06:47:38.215961856+00:00
min,2017-08-18 00:00:00+00:00,2017-08-18 00:14:59.999000+00:00,NaN,NaN,1.088600e-01,1.156100e-01,1.012900e-01,1.089600e-01,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,-2.909802e-01,-2.904870e-01,0.000000e+00,-1.000000e+00,-1.000000e+00,-3.438718e-01,2017-08-18 00:00:00+00:00
25%,2020-07-19 11:56:15+00:00,2020-07-19 12:11:14.999000064+00:00,NaN,NaN,2.440600e+00,2.449600e+00,2.431075e+00,2.440600e+00,7.885427e+02,8.907538e+05,1.556000e+03,3.842152e+02,4.388969e+05,-1.765619e-03,-1.763864e-03,2.678753e-03,-2.970293e-01,-2.969555e-01,-1.767179e-03,2020-07-19 00:00:00+00:00
50%,2022-07-23 05:52:30+00:00,2022-07-23 06:07:29.999000064+00:00,NaN,NaN,2.727000e+02,2.736600e+02,2.717450e+02,2.726900e+02,6.742888e+03,3.091837e+06,5.024000e+03,3.303426e+03,1.505831e+06,0.000000e+00,0.000000e+00,4.393900e-03,-3.226461e-02,-3.214421e-02,0.000000e+00,2022-07-23 00:00:00+00:00
75%,2024-07-12 02:48:45+00:00,2024-07-12 03:03:44.999000064+00:00,NaN,NaN,6.479000e+03,6.490562e+03,6.465020e+03,6.478950e+03,2.334692e+05,9.618265e+06,1.399700e+04,1.123581e+05,4.695050e+06,1.805967e-03,1.802460e-03,7.356588e-03,3.685075e-01,3.684703e-01,1.804338e-03,2024-07-12 00:00:00+00:00
max,2026-06-30 23:45:00+00:00,2026-06-30 23:59:59.999000+00:00,NaN,NaN,1.260112e+05,1.261996e+05,1.256480e+05,1.260112e+05,6.381015e+08,1.086850e+09,1.754730e+06,2.696371e+08,5.619115e+08,2.376784e-01,1.114000e+00,1.142917e+00,inf,inf,2.132374e-01,2026-06-30 00:00:00+00:00


### Realized Variance Calculation

In [ ]:
drive_df = (
    drive_df
    .sort_values(["symbol", "open_time"])
    .reset_index(drop=True)
)

drive_df["squared_intraday_return"] = (
    drive_df["log_return"] ** 2
)

daily_df = (
    drive_df.groupby(["symbol", "date"])
    .agg(
        realized_variance=(
            "squared_intraday_return",
            "sum"
        ),
        daily_open=("open", "first"),
        daily_high=("high", "max"),
        daily_low=("low", "min"),
        daily_close=("close", "last"),
        daily_volume=("base_volume", "sum"),
        candle_count=("open_time", "count"),
    )
    .reset_index()
)

daily_df["realized_volatility"] = np.sqrt(
    daily_df["realized_variance"]
)

EPSILON = 1e-12

daily_df["log_realized_variance"] = np.log(
    daily_df["realized_variance"] + EPSILON
)

In [ ]:
daily_df.head()

,symbol,date,realized_variance,daily_open,daily_high,daily_low,daily_close,daily_volume,candle_count,realized_volatility,log_realized_variance
0,BTCUSDT,2017-08-18 00:00:00+00:00,0.003691,4285.08,4371.52,3938.77,4108.37,1199.888264,96,0.060750,-5.601972
1,BTCUSDT,2017-08-19 00:00:00+00:00,0.005031,4108.37,4184.69,3850.00,4139.98,381.309763,96,0.070927,-5.292204
2,BTCUSDT,2017-08-20 00:00:00+00:00,0.001959,4120.98,4211.08,4032.62,4086.29,467.083022,96,0.044264,-6.235160
3,BTCUSDT,2017-08-21 00:00:00+00:00,0.004931,4069.13,4119.62,3911.79,4016.00,691.743060,96,0.070222,-5.312201
4,BTCUSDT,2017-08-22 00:00:00+00:00,0.021132,4016.00,4104.82,3400.00,4040.00,966.684858,96,0.145369,-3.856958


### Creating Initial features

In [ ]:
daily_df = (
    daily_df
    .sort_values(["symbol", "date"])
    .reset_index(drop=True)
)

previous_date = (
    daily_df.groupby("symbol")["date"]
    .shift(1)
)

previous_daily_close = (
    daily_df.groupby("symbol")["daily_close"]
    .shift(1)
)

raw_daily_return = np.log(
    daily_df["daily_close"]
    / previous_daily_close
)

previous_day_is_consecutive = (
    daily_df["date"] - previous_date
).eq(pd.Timedelta(days=1))

daily_df["daily_return"] = raw_daily_return.where(
    previous_day_is_consecutive
)

daily_df["absolute_daily_return"] = (
    daily_df["daily_return"].abs()
)

daily_df["high_low_range"] = np.log(
    daily_df["daily_high"]
    / daily_df["daily_low"]
)

daily_df["log_volume"] = np.log1p(
    daily_df["daily_volume"]
)

In [ ]:
grouped_rv = daily_df.groupby(
    "symbol"
)["realized_variance"]

daily_df["rv_mean_7d"] = grouped_rv.transform(
    lambda x: x.rolling(
        window=7,
        min_periods=7
    ).mean()
)

daily_df["rv_mean_30d"] = grouped_rv.transform(
    lambda x: x.rolling(
        window=30,
        min_periods=30
    ).mean()
)

In [ ]:
date_6_rows_ago = (
    daily_df.groupby("symbol")["date"]
    .shift(6)
)

date_29_rows_ago = (
    daily_df.groupby("symbol")["date"]
    .shift(29)
)

valid_7d_window = (
    daily_df["date"] - date_6_rows_ago
).eq(pd.Timedelta(days=6))

valid_30d_window = (
    daily_df["date"] - date_29_rows_ago
).eq(pd.Timedelta(days=29))

daily_df.loc[
    ~valid_7d_window,
    "rv_mean_7d"
] = np.nan

daily_df.loc[
    ~valid_30d_window,
    "rv_mean_30d"
] = np.nan

In [ ]:
# Build an exact next-calendar-day target lookup.

target_lookup = (
    daily_df[
        [
            "symbol",
            "date",
            "log_realized_variance",
            "realized_variance",
        ]
    ]
    .rename(
        columns={
            "date": "target_date",
            "log_realized_variance": "target_log_rv",
            "realized_variance": "target_rv",
        }
    )
)

daily_df["target_date"] = (
    daily_df["date"]
    + pd.Timedelta(days=1)
)

daily_df = daily_df.merge(
    target_lookup,
    on=["symbol", "target_date"],
    how="left",
    validate="one_to_one"
)

In [ ]:
available_targets = daily_df.loc[
    daily_df["target_log_rv"].notna()
].copy()

assert (
    available_targets["target_date"]
    - available_targets["date"]
).eq(pd.Timedelta(days=1)).all()

print(
    "Rows with valid next-day target:",
    len(available_targets)
)

print(
    "Rows without a next-day target:",
    daily_df["target_log_rv"].isna().sum()
)

Rows with valid next-day target: 11403
Rows without a next-day target: 99


In [ ]:
daily_df.head(35)

,symbol,date,realized_variance,daily_open,daily_high,daily_low,daily_close,daily_volume,candle_count,realized_volatility,log_realized_variance,daily_return,absolute_daily_return,high_low_range,log_volume,rv_mean_7d,rv_mean_30d,target_date,target_log_rv,target_rv
0,BTCUSDT,2017-08-18 00:00:00+00:00,0.003691,4285.08,4371.52,3938.77,4108.37,1199.888264,96,0.060750,-5.601972,NaN,NaN,0.104242,7.090817,NaN,NaN,2017-08-19 00:00:00+00:00,-5.292204,0.005031
1,BTCUSDT,2017-08-19 00:00:00+00:00,0.005031,4108.37,4184.69,3850.00,4139.98,381.309763,96,0.070927,-5.292204,0.007665,0.007665,0.083359,5.946231,NaN,NaN,2017-08-20 00:00:00+00:00,-6.235160,0.001959
2,BTCUSDT,2017-08-20 00:00:00+00:00,0.001959,4120.98,4211.08,4032.62,4086.29,467.083022,96,0.044264,-6.235160,-0.013053,0.013053,0.043303,6.148646,NaN,NaN,2017-08-21 00:00:00+00:00,-5.312201,0.004931
3,BTCUSDT,2017-08-21 00:00:00+00:00,0.004931,4069.13,4119.62,3911.79,4016.00,691.743060,96,0.070222,-5.312201,-0.017351,0.017351,0.051766,6.540659,NaN,NaN,2017-08-22 00:00:00+00:00,-3.856958,0.021132
4,BTCUSDT,2017-08-22 00:00:00+00:00,0.021132,4016.00,4104.82,3400.00,4040.00,966.684858,96,0.145369,-3.856958,0.005958,0.005958,0.188386,6.874906,NaN,NaN,2017-08-23 00:00:00+00:00,-5.924016,0.002674
5,BTCUSDT,2017-08-23 00:00:00+00:00,0.002674,4040.00,4265.80,4013.89,4114.01,1001.136565,96,0.051715,-5.924016,0.018154,0.018154,0.060869,6.909890,NaN,NaN,2017-08-24 00:00:00+00:00,-5.969525,0.002555
6,BTCUSDT,2017-08-24 00:00:00+00:00,0.002555,4147.00,4371.68,4085.01,4316.01,787.418753,96,0.050552,-5.969525,0.047933,0.047933,0.067823,6.670029,0.005996,NaN,2017-08-25 00:00:00+00:00,-5.958407,0.002584
7,BTCUSDT,2017-08-25 00:00:00+00:00,0.002584,4316.01,4453.91,4247.48,4280.68,573.612740,96,0.050833,-5.958407,-0.008219,0.008219,0.047456,6.353696,0.005838,NaN,2017-08-26 00:00:00+00:00,-6.322009,0.001796
8,BTCUSDT,2017-08-26 00:00:00+00:00,0.001796,4280.71,4367.00,4212.41,4337.44,228.108068,96,0.042383,-6.322009,0.013172,0.013172,0.036041,5.434194,0.005376,NaN,2017-08-27 00:00:00+00:00,-6.245614,0.001939
9,BTCUSDT,2017-08-27 00:00:00+00:00,0.001939,4332.51,4400.00,4285.54,4310.01,350.692585,96,0.044033,-6.245614,-0.006344,0.006344,0.026358,5.862757,0.005373,NaN,2017-08-28 00:00:00+00:00,-5.608642,0.003666


### Assigning train/val/test period

In [ ]:
TRAIN_END = pd.Timestamp(
    "2024-06-30",
    tz="UTC"
)

VALIDATION_START = pd.Timestamp(
    "2024-07-01",
    tz="UTC"
)

VALIDATION_END = pd.Timestamp(
    "2025-06-30",
    tz="UTC"
)

TEST_START = pd.Timestamp(
    "2025-07-01",
    tz="UTC"
)

TEST_END = pd.Timestamp(
    "2026-06-30",
    tz="UTC"
)

In [ ]:
def assign_split(target_date):

    if pd.isna(target_date):
        return None

    if target_date <= TRAIN_END:
        return "train"

    if VALIDATION_START <= target_date <= VALIDATION_END:
        return "validation"

    if TEST_START <= target_date <= TEST_END:
        return "test"

    return "outside"


daily_df["split"] = (
    daily_df["target_date"]
    .apply(assign_split)
)

In [ ]:
daily_df.groupby("split").agg(
    start_target_date=("target_date", "min"),
    end_target_date=("target_date", "max"),
    number_of_rows=("target_date", "count"),
)

,start_target_date,end_target_date,number_of_rows
split,,,
outside,2026-07-01 00:00:00+00:00,2026-07-01 00:00:00+00:00,4
test,2025-07-01 00:00:00+00:00,2026-06-30 00:00:00+00:00,1460
train,2017-08-19 00:00:00+00:00,2024-06-30 00:00:00+00:00,8578
validation,2024-07-01 00:00:00+00:00,2025-06-30 00:00:00+00:00,1460


In [ ]:
print(daily_df.columns)

FEATURE_COLUMNS = [
    "log_realized_variance",
    "daily_return",
    "absolute_daily_return",
    "high_low_range",
    "log_volume",
    "rv_mean_7d",
    "rv_mean_30d",
]

LOOKBACK_DAYS = 30

Index(['symbol', 'date', 'realized_variance', 'daily_open', 'daily_high',
       'daily_low', 'daily_close', 'daily_volume', 'candle_count',
       'realized_volatility', 'log_realized_variance', 'daily_return',
       'absolute_daily_return', 'high_low_range', 'log_volume', 'rv_mean_7d',
       'rv_mean_30d', 'target_date', 'target_log_rv', 'target_rv', 'split'],
      dtype='object')


In [ ]:
### Date - Continuity Check

def create_sequences(
    asset_df,
    feature_columns,
    lookback_days=30
):
    asset_df = (
        asset_df
        .sort_values("date")
        .reset_index(drop=True)
        .copy()
    )

    X = []
    y = []
    metadata = []

    feature_values = asset_df[
        feature_columns
    ].to_numpy(dtype=np.float32)

    target_values = asset_df[
        "target_log_rv"
    ].to_numpy(dtype=np.float32)

    for end_idx in range(
        lookback_days - 1,
        len(asset_df)
    ):
        start_idx = (
            end_idx - lookback_days + 1
        )

        sequence = feature_values[
            start_idx:end_idx + 1
        ]

        target = target_values[end_idx]

        sequence_dates = asset_df.loc[
            start_idx:end_idx,
            "date"
        ]

        # Input must consist of 30 consecutive calendar days.
        date_differences = (
            sequence_dates.diff().dropna()
        )

        if not date_differences.eq(
            pd.Timedelta(days=1)
        ).all():
            continue

        if np.isnan(sequence).any():
            continue

        if np.isnan(target):
            continue

        sequence_end_date = asset_df.loc[
            end_idx,
            "date"
        ]

        target_date = asset_df.loc[
            end_idx,
            "target_date"
        ]

        # Target must be exactly the following calendar day.
        if (
            target_date - sequence_end_date
            != pd.Timedelta(days=1)
        ):
            continue

        split = asset_df.loc[
            end_idx,
            "split"
        ]

        if split not in {
            "train",
            "validation",
            "test"
        }:
            continue

        X.append(sequence)
        y.append(target)

        metadata.append({
            "symbol": asset_df.loc[
                end_idx,
                "symbol"
            ],
            "sequence_start_date": asset_df.loc[
                start_idx,
                "date"
            ],
            "sequence_end_date": sequence_end_date,
            "target_date": target_date,
            "split": split,
            "actual_rv": asset_df.loc[
                end_idx,
                "target_rv"
            ],
        })

    return (
        np.asarray(X, dtype=np.float32),
        np.asarray(y, dtype=np.float32),
        pd.DataFrame(metadata)
    )

In [ ]:
btc_df = daily_df.loc[
    daily_df["symbol"] == "BTCUSDT"
].copy()

X, y, sequence_metadata = create_sequences(
    asset_df=btc_df,
    feature_columns=FEATURE_COLUMNS,
    lookback_days=LOOKBACK_DAYS,
)

assert (
    sequence_metadata["target_date"]
    - sequence_metadata["sequence_end_date"]
).eq(pd.Timedelta(days=1)).all()

In [ ]:
train_mask = (
    sequence_metadata["split"] == "train"
).to_numpy()

validation_mask = (
    sequence_metadata["split"] == "validation"
).to_numpy()

test_mask = (
    sequence_metadata["split"] == "test"
).to_numpy()

X_train = X[train_mask]
y_train = y[train_mask]

X_val = X[validation_mask]
y_val = y[validation_mask]

X_test = X[test_mask]
y_test = y[test_mask]

In [ ]:
print(btc_df.shape)
print(X.shape)
print(y.shape)
print(sequence_metadata.shape)

(3205, 21)
(2033, 30, 7)
(2033,)
(2033, 6)


### Final Checks

In [ ]:
assert sequence_metadata.loc[
    sequence_metadata["split"] == "train",
    "target_date"
].max() <= TRAIN_END

assert sequence_metadata.loc[
    sequence_metadata["split"] == "validation",
    "target_date"
].min() >= VALIDATION_START

assert sequence_metadata.loc[
    sequence_metadata["split"] == "validation",
    "target_date"
].max() <= VALIDATION_END

assert sequence_metadata.loc[
    sequence_metadata["split"] == "test",
    "target_date"
].min() >= TEST_START

assert sequence_metadata.loc[
    sequence_metadata["split"] == "test",
    "target_date"
].max() <= TEST_END

### Scaling

In [ ]:
n_train, lookback_days, n_features = (
    X_train.shape
)

scaler = StandardScaler()

X_train_2d = X_train.reshape(
    -1,
    n_features
)

scaler.fit(X_train_2d)

X_train_scaled = scaler.transform(
    X_train_2d
).reshape(
    X_train.shape
)

X_val_scaled = scaler.transform(
    X_val.reshape(-1, n_features)
).reshape(
    X_val.shape
)

X_test_scaled = scaler.transform(
    X_test.reshape(-1, n_features)
).reshape(
    X_test.shape
)

In [ ]:
print(
    "Training feature means:",
    X_train_scaled.mean(axis=(0, 1))
)

print(
    "Training feature standard deviations:",
    X_train_scaled.std(axis=(0, 1))
)

print("X_train_scaled:", X_train_scaled.shape)
print("X_val_scaled:", X_val_scaled.shape)
print("X_test_scaled:", X_test_scaled.shape)

Training feature means: [ 5.5625816e-07 -2.5795362e-07  8.8438867e-07  8.1631373e-07
 -8.2051002e-07 -1.3006132e-07  4.3656078e-06]
Training feature standard deviations: [1.0000149 0.9999996 0.9999903 1.0000026 0.9999963 0.9999979 0.9999932]
X_train_scaled: (1303, 30, 7)
X_val_scaled: (365, 30, 7)
X_test_scaled: (365, 30, 7)


### Saving Files

In [37]:
model_data_dir = (
    shared_project_dir
    / "data"
    / "model_ready"
)

model_data_dir.mkdir(
    parents=True,
    exist_ok=True
)

btc_output_dir = (
    model_data_dir
    / "BTCUSDT"
)

btc_output_dir.mkdir(
    parents=True,
    exist_ok=True
)

print(btc_output_dir)

/content/drive/MyDrive/Quant Research/data/model_ready/BTCUSDT


In [38]:
daily_data_path = (
    model_data_dir
    / "crypto_daily_features_v1.parquet"
)

daily_df.to_parquet(
    daily_data_path,
    index=False
)

In [39]:
scaler_path = (
    btc_output_dir
    / "feature_scaler.joblib"
)

joblib.dump(
    scaler,
    scaler_path
)

['/content/drive/MyDrive/Quant Research/data/model_ready/BTCUSDT/feature_scaler.joblib']

In [40]:
np.save(
    btc_output_dir / "X_train_scaled.npy",
    X_train_scaled
)

np.save(
    btc_output_dir / "X_val_scaled.npy",
    X_val_scaled
)

np.save(
    btc_output_dir / "X_test_scaled.npy",
    X_test_scaled
)

np.save(
    btc_output_dir / "y_train.npy",
    y_train
)

np.save(
    btc_output_dir / "y_val.npy",
    y_val
)

np.save(
    btc_output_dir / "y_test.npy",
    y_test
)

In [41]:
np.save(
    btc_output_dir / "X_train_raw.npy",
    X_train
)

np.save(
    btc_output_dir / "X_val_raw.npy",
    X_val
)

np.save(
    btc_output_dir / "X_test_raw.npy",
    X_test
)

In [42]:
train_metadata = (
    sequence_metadata.loc[
        sequence_metadata["split"] == "train"
    ]
    .reset_index(drop=True)
)

validation_metadata = (
    sequence_metadata.loc[
        sequence_metadata["split"]
        == "validation"
    ]
    .reset_index(drop=True)
)

test_metadata = (
    sequence_metadata.loc[
        sequence_metadata["split"] == "test"
    ]
    .reset_index(drop=True)
)

In [43]:
train_metadata.to_parquet(
    btc_output_dir / "train_metadata.parquet",
    index=False
)

validation_metadata.to_parquet(
    btc_output_dir / "validation_metadata.parquet",
    index=False
)

test_metadata.to_parquet(
    btc_output_dir / "test_metadata.parquet",
    index=False
)

### Saving Experiment Configs

In [44]:
configuration = {
    "asset": "BTCUSDT",
    "candle_interval": "15m",
    "forecast_horizon_days": 1,
    "lookback_days": LOOKBACK_DAYS,
    "target": "log_realized_variance",
    "feature_columns": FEATURE_COLUMNS,
    "train_end": str(TRAIN_END),
    "validation_start": str(
        VALIDATION_START
    ),
    "validation_end": str(
        VALIDATION_END
    ),
    "test_start": str(TEST_START),
    "test_end": str(TEST_END),
    "X_train_shape": list(
        X_train_scaled.shape
    ),
    "X_validation_shape": list(
        X_val_scaled.shape
    ),
    "X_test_shape": list(
        X_test_scaled.shape
    ),
}

configuration_path = (
    btc_output_dir
    / "dataset_configuration.json"
)

with open(
    configuration_path,
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        configuration,
        file,
        indent=2
    )

### Final Checks

In [45]:
saved_files = sorted(
    path.name
    for path in btc_output_dir.iterdir()
)

for filename in saved_files:
    print(filename)

X_test_raw.npy
X_test_scaled.npy
X_train_raw.npy
X_train_scaled.npy
X_val_raw.npy
X_val_scaled.npy
dataset_configuration.json
feature_scaler.joblib
test_metadata.parquet
train_metadata.parquet
validation_metadata.parquet
y_test.npy
y_train.npy
y_val.npy


In [46]:
loaded_X_train = np.load(
    btc_output_dir
    / "X_train_scaled.npy"
)

loaded_y_train = np.load(
    btc_output_dir
    / "y_train.npy"
)

loaded_scaler = joblib.load(
    btc_output_dir
    / "feature_scaler.joblib"
)

loaded_train_metadata = pd.read_parquet(
    btc_output_dir
    / "train_metadata.parquet"
)

assert loaded_X_train.shape == X_train_scaled.shape
assert loaded_y_train.shape == y_train.shape

assert (
    len(loaded_X_train)
    == len(loaded_y_train)
    == len(loaded_train_metadata)
)

print("All stored outputs verified successfully.")

All stored outputs verified successfully.


In [47]:
print(X_train_scaled.shape)
print(X_val_scaled.shape)
print(X_test_scaled.shape)

display(
    sequence_metadata.groupby("split").agg(
        samples=("target_date", "size"),
        first_target=("target_date", "min"),
        last_target=("target_date", "max"),
    )
)

(1303, 30, 7)
(365, 30, 7)
(365, 30, 7)


,samples,first_target,last_target
split,,,
test,365,2025-07-01 00:00:00+00:00,2026-06-30 00:00:00+00:00
train,1303,2017-11-05 00:00:00+00:00,2024-06-30 00:00:00+00:00
validation,365,2024-07-01 00:00:00+00:00,2025-06-30 00:00:00+00:00
